Prompting with Prompt Templates for LLM Input / Output with LangChain

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [5]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


Chat Models and LLMs

Large Language Models (LLMs) are a core component of LangChain. LangChain does not implement or build its own LLMs. It provides a standard API for interacting with almost every LLM out there.

There are lots of LLM providers (OpenAI, Hugging Face, etc) - the LLM class is designed to provide a standard interface for all of them.

Accessing Commercial LLMs like ChatGPT

In [6]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0)

Prompt Templates

Prompt templates are pre-designed formats used to generate prompts for language models. These templates can include instructions, few-shot examples, and specific contexts and questions suited for particular tasks.

LangChain provides tools for creating and using prompt templates. It aims to develop model-agnostic templates to facilitate the reuse of existing templates across different language models. Typically, these models expect prompts in the form of either a string or a list of chat messages.

Types of Prompt Templates

PromptTemplate:

Used for creating string-based prompts.
Utilizes Python's str.format syntax for templating, supporting any number of variables, including scenarios with no variables.
ChatPromptTemplate:

Designed for chat models, where the prompt consists of a list of chat messages.
Each chat message includes content and a role parameter. For instance, in the OpenAI Chat Completions API, a chat message could be assigned to an AI assistant, a human, or a system role.
FewShotChatMessagePromptTemplate

A few-shot prompt template can be constructed from a set of examples

PromptTemplate

We can use PromptTemplate to create a template for a string prompt.

By default, PromptTemplate uses Python's str.format syntax for templating.

You can create custom prompt templates that format the prompt in any way you want. For more information, see Prompt Template Composition.

In [7]:
from langchain_core.prompts import PromptTemplate
# Simple prompt

prompt = """Explain to me what is Generative AI in 3 bullet points?"""
prompt_template = PromptTemplate.from_template(prompt)
prompt_template

PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Explain to me what is Generative AI in 3 bullet points?')

In [8]:
prompt_template.format()

'Explain to me what is Generative AI in 3 bullet points?'

In [9]:
response = llm.invoke(prompt_template.format())
print(response.content)

- **Definition**: Generative AI refers to a class of artificial intelligence models that can create new content, such as text, images, music, or videos, by learning patterns and structures from existing data.

- **Techniques**: It often employs advanced techniques like deep learning, particularly neural networks such as Generative Adversarial Networks (GANs) and transformer models, to generate high-quality and coherent outputs.

- **Applications**: Generative AI is used in various fields, including creative arts (e.g., art and music generation), content creation (e.g., writing and journalism), and even in practical applications like drug discovery and data augmentation.


Advantage of using prompt template is we can have some placeholder in prompt and values of placeholders will be coming from some realtime operations.

In [11]:
# more complex prompt with placeholders
prompt = """Explain to me briefly about {topic} in {language} in {numOfWords} words."""

prompt_template = PromptTemplate.from_template(prompt)
prompt_template

PromptTemplate(input_variables=['language', 'numOfWords', 'topic'], input_types={}, partial_variables={}, template='Explain to me briefly about {topic} in {language} in {numOfWords} words.')

In [15]:
inputs = [("Generative AI", "english"),
          ("Artificial Intelligence", "hindi"),
          ("Deep Learning", "german")]
prompts = [prompt_template.format(topic = topic,language = language, numOfWords = 50) for topic, language in inputs]
prompts

['Explain to me briefly about Generative AI in english in 50 words.',
 'Explain to me briefly about Artificial Intelligence in hindi in 50 words.',
 'Explain to me briefly about Deep Learning in german in 50 words.']

In [17]:
# use map to run on multiple prompts in one go
response = llm.map().invoke(prompts)

In [18]:
response

[AIMessage(content='Generative AI refers to algorithms that create new content, such as text, images, music, or videos, by learning patterns from existing data. It uses techniques like deep learning and neural networks to generate original outputs, enabling applications in art, entertainment, and various industries, enhancing creativity and automation.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 22, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_087aedc2a2', 'finish_reason': 'stop', 'logprobs': None}, id='run--7c43abe4-3612-4a53-bd26-198eb0a50696-0', usage_metadata={'input_tokens': 22, 'output_tokens': 59, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_read':

In [19]:
for res in response:
    print(res.content)
    print("-------")

Generative AI refers to algorithms that create new content, such as text, images, music, or videos, by learning patterns from existing data. It uses techniques like deep learning and neural networks to generate original outputs, enabling applications in art, entertainment, and various industries, enhancing creativity and automation.
-------
कृत्रिम बुद्धिमत्ता (AI) एक तकनीक है जो मशीनों को मानव जैसी सोचने, सीखने और निर्णय लेने की क्षमता देती है। यह डेटा का विश्लेषण करके समस्याओं को हल करने, स्वचालन, और विभिन्न कार्यों में सहायता करने में सक्षम है, जैसे कि भाषा पहचान, छवि विश्लेषण, और गेम खेलना।
-------
Deep Learning ist ein Teilbereich des maschinellen Lernens, der auf künstlichen neuronalen Netzen basiert. Es ermöglicht Computern, komplexe Muster in großen Datenmengen zu erkennen. Anwendungen finden sich in der Bild- und Spracherkennung, der natürlichen Sprachverarbeitung und autonomen Systemen. Deep Learning revolutioniert viele Branchen durch seine Leistungsfähigkeit.
-------


ChatPromptTemplate

The standard prompt format to chat models is a list of chat messages.

Each chat message is associated with content, and an additional parameter called role. For example, in the OpenAI Chat Completions API, a chat message can be associated with an AI assistant, a human or a system role.

In [21]:
from langchain_core.prompts import ChatPromptTemplate
# simple prompt with placeholders
prompt = """Explain to me briefly about {topic} in {numOfWords} words."""
chat_template = ChatPromptTemplate.from_template(prompt)
chat_template

ChatPromptTemplate(input_variables=['numOfWords', 'topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['numOfWords', 'topic'], input_types={}, partial_variables={}, template='Explain to me briefly about {topic} in {numOfWords} words.'), additional_kwargs={})])

In [23]:
topics = ['motgage', 'frractional real estate', 'commercial real estate']
prompts = [chat_template.format(topic=topic, numOfWords = 30) for topic in topics]
prompts

['Human: Explain to me briefly about motgage in 30 words.',
 'Human: Explain to me briefly about frractional real estate in 30 words.',
 'Human: Explain to me briefly about commercial real estate in 30 words.']

In [25]:
responses = llm.map().invoke(prompts)

In [26]:
responses

[AIMessage(content='A mortgage is a loan used to purchase real estate, where the property serves as collateral. Borrowers repay the loan with interest over a specified term, typically 15 to 30 years.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 21, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c2953e649', 'finish_reason': 'stop', 'logprobs': None}, id='run--fb4eb090-dbd7-47d2-8efa-7a0e4578113e-0', usage_metadata={'input_tokens': 21, 'output_tokens': 38, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content='Fractional real estate allows multiple investors to share ownership of a p

In [29]:
for res in responses:
    print(res.content)
    print("----")

A mortgage is a loan used to purchase real estate, where the property serves as collateral. Borrowers repay the loan with interest over a specified term, typically 15 to 30 years.
----
Fractional real estate allows multiple investors to share ownership of a property, dividing costs and benefits. It offers access to real estate investments with lower financial commitment and shared responsibilities.
----
Commercial real estate involves properties used for business purposes, including office buildings, retail spaces, warehouses, and industrial sites. It generates income through leasing and is a key investment sector.
----


In [30]:
# more complex prompt with a series of messages
messages = [
        ("system", "Act as an expert in real estate and provide brief answers in 25 words "),
        ("human", "what is your name?"),
        ("ai", "my name is AIBot"),
        ("human", "{user_prompt}"),
]
chat_template = ChatPromptTemplate.from_messages(messages)
chat_template

ChatPromptTemplate(input_variables=['user_prompt'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Act as an expert in real estate and provide brief answers in 25 words '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='what is your name?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='my name is AIBot'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_prompt'], input_types={}, partial_variables={}, template='{user_prompt}'), additional_kwargs={})])

In [31]:
text_prompts = ["what is your name?",
                "explain commercial real estate to me"]
chat_prompts = [chat_template.format(user_prompt=prompt) for prompt in text_prompts]
chat_prompts

['System: Act as an expert in real estate and provide brief answers in 25 words \nHuman: what is your name?\nAI: my name is AIBot\nHuman: what is your name?',
 'System: Act as an expert in real estate and provide brief answers in 25 words \nHuman: what is your name?\nAI: my name is AIBot\nHuman: explain commercial real estate to me']

In [32]:
print(chat_prompts[0])

System: Act as an expert in real estate and provide brief answers in 25 words 
Human: what is your name?
AI: my name is AIBot
Human: what is your name?


In [33]:
print(chat_prompts[1])

System: Act as an expert in real estate and provide brief answers in 25 words 
Human: what is your name?
AI: my name is AIBot
Human: explain commercial real estate to me


In [34]:
responses = llm.map().invoke(chat_prompts)
for response in responses:
  print(response.content)
  print('-----')

AI: I am AIBot, your real estate expert here to assist you with any property-related questions or advice you may need.
-----
Commercial real estate involves properties used for business purposes, including office buildings, retail spaces, warehouses, and industrial sites, generating income through leasing or sales.
-----


PromptTemplate and ChatPromptTemplate supports LCEL

PromptTemplate and ChatPromptTemplate implement the Runnable interface, the basic building block of the LangChain Expression Language (LCEL). This means they support invoke, ainvoke, stream, astream, batch, abatch, astream_log calls.

PromptTemplate accepts a dictionary (of the prompt variables) and returns a StringPromptValue. A ChatPromptTemplate accepts a dictionary and returns a ChatPromptValue.

FewShotChatMessagePromptTemplate

This is a Chat prompt template that supports few-shot examples.

The high level structure of produced by this prompt template is a list of messages consisting of prefix message(s), example message(s), and suffix message(s).

This structure enables creating a conversation with intermediate examples like:

System: You are a helpful AI Assistant
Human: What is 2+2?
AI: 4
Human: What is 2+3?
AI: 5
Human: What is 4+4?






Define the few-shot examples you'd like to include.

In [36]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate


In [37]:
example_data = [
    ("""Wimbledon 2023 Women’s Final.
    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title""",
     'Sports'),
    ("""It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships
    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay
    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of
    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action""",
     'Sports'),
    ("""Kicking things off with the Galaxy Tab S9 FE Plus. It's going to be powered by the Exynos 1380
    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and
    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,
    Light Pink, and Silver color options.""",
     'Technology'),
    ("""The share of the cigarette business to ITC’s business in revenue terms has dropped from 47
    per cent in FY13 to 37 per cent in FY23 on a larger revenue base.
    According to the company’s most recent annual report, the leadership position has been sustained
    "through its unwavering focus on nurturing a future-ready portfolio of world-class products anchored on
    its integrated seed to smoke value chain, superior consumer insights, robust innovation pipeline and
    world-class product development capabilities." """,
     'Business')
]
example_data

[('Wimbledon 2023 Women’s Final.\n    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title',
  'Sports'),
 ("It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships\n    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay\n    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of\n    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action",
  'Sports'),
 ("Kicking things off with the Galaxy Tab S9 FE Plus. It's going to be powered by the Exynos 1380\n    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and\n    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,\n    Light Pink, and Silver color options.",
  'Technology'),
 ('The share of the cigarette business to ITC’s business in revenue terms has dropped f

In [38]:
example_data_formatter = [{"input":input, "output":output} for input, output in example_data]
example_data_formatter

[{'input': 'Wimbledon 2023 Women’s Final.\n    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title',
  'output': 'Sports'},
 {'input': "It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships\n    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay\n    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of\n    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action",
  'output': 'Sports'},
 {'input': "Kicking things off with the Galaxy Tab S9 FE Plus. It's going to be powered by the Exynos 1380\n    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and\n    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,\n    Light Pink, and Silver color options.",
  'output': 'Technology'},
 {'input': 'The share of the c

Assemble them into the few-shot prompt template.

In [41]:
data_prompt = [
    ("human","{input}"),
    ("ai","{output}")
]
data_prompt

[('human', '{input}'), ('ai', '{output}')]

In [42]:
chat_prompts_few_shot = ChatPromptTemplate.from_messages(data_prompt)
chat_prompts_few_shot

ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], input_types={}, partial_variables={}, template='{output}'), additional_kwargs={})])

In [44]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = chat_prompts_few_shot,
    examples = example_data_formatter
)
few_shot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'Wimbledon 2023 Women’s Final.\n    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title', 'output': 'Sports'}, {'input': "It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships\n    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay\n    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of\n    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action", 'output': 'Sports'}, {'input': "Kicking things off with the Galaxy Tab S9 FE Plus. It's going to be powered by the Exynos 1380\n    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and\n    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,\n    Light Pink, and Silver color options.", 'output': 'Technology

In [45]:
print(few_shot_prompt.format())

Human: Wimbledon 2023 Women’s Final.
    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title
AI: Sports
Human: It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships
    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay
    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of
    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action
AI: Sports
Human: Kicking things off with the Galaxy Tab S9 FE Plus. It's going to be powered by the Exynos 1380
    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and
    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,
    Light Pink, and Silver color options.
AI: Technology
Human: The share of the cigarette business to ITC’s business in revenue terms has dropped from

In [57]:
final_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Classify every new news article using a similar format as shown below"),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt_template

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Classify every new news article using a similar format as shown below'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': 'Wimbledon 2023 Women’s Final.\n    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title', 'output': 'Sports'}, {'input': "It's time to call it a day as India had another medal-laden day at the Asian Athletics Championships\n    in Bangkok. Murali Sreeshankar's Olympic qualifying jump grabbed the limelight while 400m mixed relay\n    team added's India's sixth gold medal! There will be plenty of medals up for grab on the final day of\n    the tournament as well with Jyothi Yarraji, women's and men's 400m relay teams in action", 'output': 'Sports'}, {'input': "Kicking things off with the Galaxy Tab S9 FE Plus. It'

In [63]:
docs = ["""Arsenal finally completed the signing of Rice as they announced the player's arrival on Saturday.
Right after completing his move, Rice revealed that he is a great admirer of Arteta
and that the Spaniard's presence in the Arsenal dressing room is one of the main reasons
why he decided to join the North London club.""",
        """A heat wave sweeping across Europe is prompting health warnings as millions
        of people endure sweltering temperatures. Italy has issued red alerts in 16 cities with
        the country's weather centre warning of record heat especially in southern regions.
        Parts of Greece are forecast to hit highs of 44 degrees celsius, while France,
        Germany, Spain and Poland are also enduring extreme temperatures.
        Scientists say greenhouse gases and the El Niño weather pattern are raising global temperatures.
        """]

In [65]:
final_prompts = [final_prompt_template.format(input=doc) for doc in docs]

In [66]:
final_prompts

['System: Classify every new news article using a similar format as shown below\nHuman: Wimbledon 2023 Women’s Final.\n    Marketa Vondrousova beats Ons Jabeur to win her maiden Grand Slam title\nAI: Sports\nHuman: It\'s time to call it a day as India had another medal-laden day at the Asian Athletics Championships\n    in Bangkok. Murali Sreeshankar\'s Olympic qualifying jump grabbed the limelight while 400m mixed relay\n    team added\'s India\'s sixth gold medal! There will be plenty of medals up for grab on the final day of\n    the tournament as well with Jyothi Yarraji, women\'s and men\'s 400m relay teams in action\nAI: Sports\nHuman: Kicking things off with the Galaxy Tab S9 FE Plus. It\'s going to be powered by the Exynos 1380\n    with 8GB of RAM, and pack a 12.4-inch display. The slate will measure 285.4 x 185.4 x 6.54 mm and\n    feature dual speakers, a side-mounted fingerprint sensor. It will come in Grey, Light Green,\n    Light Pink, and Silver color options.\nAI: Techn

In [67]:
responses = llm.map().invoke(final_prompts)

In [69]:
responses[0]

AIMessage(content='AI: Sports', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 431, 'total_tokens': 434, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_800187fee2', 'finish_reason': 'stop', 'logprobs': None}, id='run--8d1db6b7-1093-491d-ba19-83f06a3bfa07-0', usage_metadata={'input_tokens': 431, 'output_tokens': 3, 'total_tokens': 434, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [71]:
for response in responses:
    print(response.content)
    print('-----')

AI: Sports
-----
AI: Environment
-----
